<a href="https://colab.research.google.com/github/pranjalgahlot123/toxic-comment-detection/blob/main/Toxic_comment_detection_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Toxic Comment Classification: TF-IDF + Random Forest Pipeline
# ============================================================
# Approach: TF-IDF vectorization + Random Forest classifier
# Tuning: GridSearchCV over 27 hyperparameter configurations
#         × 5-fold CV = 135 model fits
# Metric: ROC-AUC
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import nltk
from nltk import WordNetLemmatizer, pos_tag, word_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import pickle

# ------------------------------------------------------------
# 1. Data Loading
# ------------------------------------------------------------
data = pd.read_csv("/content/drive/MyDrive/toxic comment/FinalBalancedDataset.csv")
data = data.drop("Unnamed: 0", axis=1)

print(f"Dataset shape: {data.shape}")
print(f"Class distribution:\n{data['Toxicity'].value_counts()}")

# ------------------------------------------------------------
# 2. Text Preprocessing (NLTK resources)
# ------------------------------------------------------------
nltk.download('punkt')
nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

wordnet_lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    """Map Treebank POS tags to WordNet POS tags for lemmatization."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def prepare_text(text):
    """Lowercase, strip non-alphabetic chars, tokenize, POS-tag, and lemmatize."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\']', ' ', text)
    text = word_tokenize(text)
    text = pos_tag(text)
    lemma = [wordnet_lemmatizer.lemmatize(i[0], pos=get_wordnet_pos(i[1])) for i in text]
    lemma = ' '.join(lemma)
    return lemma

# Apply preprocessing once and cache the cleaned text
# (Note: In the original pipeline, TF-IDF was fit on raw text; if you want
#  to keep preprocessing, apply it before passing to the pipeline.)
data['clean_tweet'] = data['tweet'].apply(prepare_text)

# ------------------------------------------------------------
# 3. Pipeline: TF-IDF + Random Forest
# ------------------------------------------------------------
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=list(nltk_stopwords.words('english')))),
    ('classifier', RandomForestClassifier(random_state=42))
])

# ------------------------------------------------------------
# 4. Hyperparameter Grid: 27 configurations
#    3 (max_features) × 3 (n_estimators) × 3 (max_depth) = 27
#    With 5-fold CV → 27 × 5 = 135 model fits
# ------------------------------------------------------------
param_grid = {
    'tfidf__max_features': [1000, 5000, 10000],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
}

n_configs = (
    len(param_grid['tfidf__max_features'])
    * len(param_grid['classifier__n_estimators'])
    * len(param_grid['classifier__max_depth'])
)
print(f"Hyperparameter configurations: {n_configs}")
print(f"Total model fits (5-fold CV): {n_configs * 5}")

# ------------------------------------------------------------
# 5. Grid Search with 5-fold CV, ROC-AUC scoring
# ------------------------------------------------------------
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(data['clean_tweet'], data['Toxicity'])

# ------------------------------------------------------------
# 6. Report Best Configuration & Score
# ------------------------------------------------------------
print("\n=== Best Configuration ===")
print("Best Parameters:", grid_search.best_params_)
print(f"Best ROC AUC (CV mean): {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

# ------------------------------------------------------------
# 7. Cross-Validation of Best Model for Robust Estimate
# ------------------------------------------------------------
cv_scores = cross_val_score(
    best_model,
    data['clean_tweet'],
    data['Toxicity'],
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

mean_auc = np.mean(cv_scores)
std_auc = np.std(cv_scores)

print("\n=== Cross-Validation Results (Best Model) ===")
print("Fold scores:", np.round(cv_scores, 4))
print(f"Mean ROC AUC: {mean_auc:.4f} ({mean_auc*100:.2f}%)")
print(f"Std Dev:      {std_auc:.4f} ({std_auc*100:.2f} pp)")

# ------------------------------------------------------------
# 8. Serialize the Optimized Pipeline as a Reusable Artifact
# ------------------------------------------------------------
with open("best_toxicity_model.pkt", "wb") as f:
    pickle.dump(best_model, f)

print("\nModel saved to 'best_toxicity_model.pkt'")